Connected to Python 3.13.2

In [ ]:
#!/usr/bin/env python3
"""KDE-MLE による POPs 年次トレンド解析（環境省「黒本」手法の再現）.

本スクリプトは、環境モニタリングデータ（POPs 等）に対して以下の 5 つの
解析を物質ごとに実行し、結果を CSV に出力する。

1. KDE-MLE（simple）        : 全残差から単一 KDE を構築するベースライン
2. KDE-MLE（iterative）     : 残差 KDE を反復更新する主軸モデル
3. KDE-MLE（MOE Fig.5）     : 年度別残差 KDE を合成する環境省図 5 準拠モデル
4. ブートストラップ検定      : 前期・後期の平均値差 95% CI
5. 二項ロジスティック回帰    : ND（不検出）割合の経年傾向

実行方法::

    cd PY_MOE_KDE_AIC/kde_mle
    python kde_mle_iterative_v1_1d.py            # data/大気温暖期.csv を解析
    python kde_mle_iterative_v1_1d.py 水質 底質   # 複数ファイルをまとめて解析

本モジュールはリファクタリング版であり、公開関数のシグネチャと数値結果は
従来版と同一に保っている。
"""

from __future__ import annotations

import argparse
import math
import os
import sys
import warnings
from typing import Callable, Optional, Sequence

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.stats import gaussian_kde
from sklearn.utils import resample
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning

try:
    from module.module_kde_mle_2a import (
        run_mle_and_aic_iterative,
        run_mle_and_aic_simple,
        calc_coef_ci_bootstrap_kde,
        calc_coef_ci_hessian,
    )
except ModuleNotFoundError:  # 別ディレクトリから実行された場合のフォールバック
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from module.module_kde_mle_2a import (
        run_mle_and_aic_iterative,
        run_mle_and_aic_simple,
        calc_coef_ci_bootstrap_kde,
        calc_coef_ci_hessian,
    )


# =====================================================================
# 定数（マジックナンバーの一元管理）
# =====================================================================

LOG_FLOOR = 1e-10
"""対数尤度計算時の 0 除算・log(0) 回避用フロア。"""

BAD_FIT_PENALTY = 1e10
"""最適化が破綻した際に返すペナルティ値（実質 -inf 相当）。"""

#ND_SUBSTITUTE_DIVISOR = np.sqrt(2.0)
ND_SUBSTITUTE_DIVISOR = 2.0
"""ND 値の代替値 = 検出下限値 / 2（環境省法）。"""

MIN_LOG_CONC_STD = 1e-6
"""log 濃度の標準偏差がこれ未満の物質は解析対象外（全 ND 等）。"""

MIN_SAMPLES = 24
"""1 物質あたりの最小サンプル数。これ未満はスキップ。"""

# --- 年度別残差 KDE（環境省 図 5）関連 ---
YEARLY_MIN_BW = 0.15
"""年度別 KDE のバンド幅の下限（データスケール）。ND 集中時のスパイク化を防ぐ。"""

YEARLY_STD_EPS = 1e-8
"""全残差の標準偏差がこれ未満なら年度別 KDE を構築しない。"""

YEARLY_FLAT_STD_EPS = 1e-4
"""年度内標準偏差がこれ未満を「平坦（ND ばかり）」とみなす閾値。"""

YEARLY_FLAT_JITTER = 1e-4
"""平坦とみなした年度に加える微小ノイズの振幅。"""

# --- 反復最尤推定 ---
DEFAULT_MAX_ITER = 50
"""反復 KDE-MLE のデフォルト最大反復回数。"""

DEFAULT_TOLERANCE = 1e-3
"""反復 KDE-MLE のデフォルト収束判定閾値。"""

MAIN_ITERATIVE_MAX_ITER = 30
"""メイン処理で iterative 版に渡す最大反復回数。"""

# --- 判定・検定 ---
P1_THRESHOLD = 0.95
"""AIC 事後確率 p1 がこれ以上なら「傾向あり」と判定する。"""

BOOTSTRAP_R = 10_000
"""ブートストラップのリサンプリング回数。"""

BOOTSTRAP_ALPHA_PERCENTILES = (2.5, 97.5)
"""ブートストラップ 95% 信頼区間のパーセンタイル。"""

# --- 入出力 ---
DEFAULT_FILENAMES: tuple[str, ...] = ("PFOS大気温暖期",)    #PFOS大気温暖期",)#大気温暖期   生物
"""引数未指定時に解析するファイル（拡張子なし）の既定値。"""

INPUT_COLUMNS: tuple[str, ...] = (
    "substance",
    "year",
    "site_id",
    "concentration",
    "detection_limit",
)
"""入力 CSV から読み込む必須カラム。"""

CSV_ENCODING = "utf-8-sig"
"""Excel での文字化けを避けるための出力エンコーディング。"""


# =====================================================================
# 汎用ユーティリティ
# =====================================================================

def clean_value_user(val: object) -> float:
    """測定値・検出下限値の文字列表現を float に正規化する.

    ``tr(...)`` の括弧や ``---``（欠測）を除去し、``nd`` / ``ND`` は 0 として
    扱う。変換できない場合は ``NaN`` を返す。

    Parameters
    ----------
    val : object
        元のセル値（文字列・数値・NaN など）。

    Returns
    -------
    float
        変換後の数値。失敗時は ``NaN``。
    """
    if isinstance(val, str):
        val = val.replace("tr(", "").replace(")", "")
        val = val.replace("---", "")
        val = val.replace("nd", "0")
        val = val.replace("ND", "0")
    try:
        if val == "":
            return np.nan
        return float(val)
    except (ValueError, TypeError):
        return np.nan


def format_ci(ci: object, digits: int = 4) -> list[float]:
    """信用区間を安全に丸めて 2 要素のリストに統一する.

    2 要素かつ有限な値であれば丸めたリストを、それ以外は
    ``[nan, nan]`` を返す。
    """
    try:
        arr = np.asarray(ci, dtype=float)
        if arr.size == 2 and not np.isnan(arr).any():
            return [round(float(arr[0]), digits), round(float(arr[1]), digits)]
    except Exception:
        pass
    return [np.nan, np.nan]


def format_float(val: object, digits: int = 4) -> float:
    """数値を安全に丸める（変換不能なら ``NaN``）。"""
    try:
        return round(float(val), digits)
    except Exception:
        return np.nan



def model_weight(ic0: float, ic1: float) -> float:
    """情報量基準（AIC / AICc）からモデル1（線形モデル）の重みを計算する."""
    diff = float(np.clip(ic0 - ic1, -500.0, 500.0))
    return float(1.0 / (1.0 + np.exp(-0.5 * diff)))


def calc_aicc(aic: float, k: int, n_samples: Optional[int]) -> float:
    """小標本補正を加えた AICc を計算する."""
    if n_samples is None:
        return np.nan
    denom = n_samples - k - 1
    if denom <= 0:
        return np.inf
    return float(aic + (2.0 * k * (k + 1.0)) / denom)


def compute_aic_posterior(
    log_likelihood_linear: float,
    log_likelihood_constant: float,
    n_params_linear: int = 2,
    n_params_constant: int = 1,
    n_samples: Optional[int] = None,
) -> tuple[float, float, float, float, float, float]:
    """AIC / AICc とそれぞれのモデル重み p1, p1_c を計算する."""
    aic1 = -2.0 * log_likelihood_linear + 2.0 * n_params_linear
    aic0 = -2.0 * log_likelihood_constant + 2.0 * n_params_constant
    p1 = model_weight(aic0, aic1)

    aicc1 = calc_aicc(aic1, n_params_linear, n_samples)
    aicc0 = calc_aicc(aic0, n_params_constant, n_samples)
    p1_c = model_weight(aicc0, aicc1)

    return aic0, aic1, p1, aicc0, aicc1, p1_c
# =====================================================================
# 環境省（図5）準拠：年度ごとの残差 KDE を合成するモデル
# =====================================================================



def build_yearly_kdes(
    residuals: np.ndarray,
    x_years: np.ndarray,
    min_bw: float = YEARLY_MIN_BW,
) -> list:
    """年度ごとに残差の KDE オブジェクトを作成してリストで返す.

    年度ごとにバンド幅を独立に決めると、ND 値が集中した年度で KDE が
    スパイク化する。これを避けるため、全残差から求めた基準バンド幅
    ``target_bw_abs``（``min_bw`` を下限とする）を全年度に共通適用する。

    Parameters
    ----------
    residuals : array-like
        現在のパラメータでの残差。
    x_years : array-like
        各サンプルの年度（中心化済みで可）。
    min_bw : float, optional
        バンド幅の下限（データスケール）。

    Returns
    -------
    list
        ``scipy.stats.gaussian_kde`` のリスト。
        有効な年度が無い場合は空リスト。
    """
    unique_years = np.unique(x_years)
    yearly_kdes = []

    # 全体残差から基準となるバンド幅（標準偏差 × factor）を算出
    overall_std = np.std(residuals)
    if overall_std < YEARLY_STD_EPS:
        return []
    base_kde = gaussian_kde(residuals)
    # 全体のバンド幅（データスケール上の幅）
    target_bw_abs = max(base_kde.factor * overall_std, min_bw)

    for yr in unique_years:
        yr_res = residuals[x_years == yr]
        if len(yr_res) < 2:
            continue

        yr_std = np.std(yr_res)
        if yr_std < YEARLY_FLAT_STD_EPS:
            # 年度内のばらつきが極端に小さい（ND ばかり）場合は、
            # 擬似的に微小ノイズを加えて全体の幅に合わせる
            yr_res_jitter = yr_res + np.linspace(
                -YEARLY_FLAT_JITTER, YEARLY_FLAT_JITTER, len(yr_res)
            )
            bw_factor = target_bw_abs / np.std(yr_res_jitter)
            yearly_kdes.append(gaussian_kde(yr_res_jitter, bw_method=bw_factor))
        else:
            bw_factor = target_bw_abs / yr_std
            yearly_kdes.append(gaussian_kde(yr_res, bw_method=bw_factor))

    return yearly_kdes


def eval_pooled_yearly_pdf(residuals: np.ndarray, yearly_kdes: list) -> np.ndarray:
    """年度別 KDE の確率密度を等重みで合成（平均化）した PDF を返す.

    図 5 の「足し合わせた共通分布」に相当する。年度 KDE が空の場合は
    ``LOG_FLOOR`` で埋めた配列を返す。
    """
    if not yearly_kdes:
        return np.full_like(residuals, LOG_FLOOR, dtype=float)

    total_pdf = np.zeros_like(residuals, dtype=float)
    for kde in yearly_kdes:
        total_pdf += kde.pdf(residuals)

    return total_pdf / len(yearly_kdes)


def neg_log_likelihood_yearly_fixed_kde(
    params: Sequence[float],
    x: np.ndarray,
    y: np.ndarray,
    yearly_kdes_fixed: list,
) -> float:
    """【線形モデル用・環境省図 5 準拠】固定 KDE リストによる負の対数尤度.

    Parameters
    ----------
    params : array-like, shape (2,)
        ``[a, b]``（傾き・切片）。
    x : array-like
        説明変数（中心化年度）。
    y : array-like
        応答変数（log 濃度）。
    yearly_kdes_fixed : list
        固定された年度別 KDE のリスト。
    """
    a, b = params
    residuals = y - (a * x + b)
    try:
        pdf_values = eval_pooled_yearly_pdf(residuals, yearly_kdes_fixed)
        log_likelihood = np.sum(np.log(pdf_values + LOG_FLOOR))
        return -log_likelihood
    except Exception:
        return BAD_FIT_PENALTY


def neg_log_likelihood_yearly_0(
    params: Sequence[float], x: np.ndarray, y: np.ndarray
) -> float:
    """【定数モデル用（傾き a=0）・環境省図 5 準拠】負の対数尤度.

    残差から都度、年度別 KDE を再構築する。
    """
    b0 = params[0]
    residuals = y - b0
    try:
        yearly_kdes = build_yearly_kdes(residuals, x)
        pdf_values = eval_pooled_yearly_pdf(residuals, yearly_kdes)
        log_likelihood = np.sum(np.log(pdf_values + LOG_FLOOR))
        return -log_likelihood
    except Exception:
        return BAD_FIT_PENALTY


# =====================================================================
# 目的関数群（KDE ベースの負の対数尤度）
# =====================================================================

def neg_log_likelihood_fixed_kde(
    params: Sequence[float],
    x: np.ndarray,
    y: np.ndarray,
    kde_fixed: gaussian_kde,
) -> float:
    """【線形モデル用】固定された KDE を用いた負の対数尤度.

    反復最尤推定の各ステップで KDE を固定し、その下でパラメータを
    最適化するために用いる。
    """
    a, b = params
    residuals = y - (a * x + b)
    try:
        pdf_values = kde_fixed.pdf(residuals)
        log_likelihood = np.sum(np.log(pdf_values + LOG_FLOOR))
        return -log_likelihood
    except Exception:
        return BAD_FIT_PENALTY


def neg_log_likelihood(params: Sequence[float], x: np.ndarray, y: np.ndarray) -> float:
    """【線形モデル用】残差から KDE を作り直して負の対数尤度を計算する."""
    a, b = params
    residuals = y - (a * x + b)
    try:
        kde = gaussian_kde(residuals)
        log_likelihood = np.sum(np.log(kde.pdf(residuals) + LOG_FLOOR))
        return -log_likelihood
    except np.linalg.LinAlgError:
        return BAD_FIT_PENALTY


def neg_log_likelihood_1(
    params: Sequence[float],
    x: np.ndarray,
    y: np.ndarray,
    initial_kde: object = None,
) -> float:
    """【互換用】``neg_log_likelihood`` の別名.

    .. note::
        ``initial_kde`` は受け取るが使用しない（従来版の挙動を維持）。
    """
    return neg_log_likelihood(params, x, y)


def neg_log_likelihood_0(
    params: Sequence[float], x: np.ndarray, y: np.ndarray
) -> float:
    """【定数モデル用】残差から KDE を作り直して負の対数尤度を計算する."""
    b0 = params[0]
    residuals = y - b0
    try:
        kde = gaussian_kde(residuals)
        log_likelihood = np.sum(np.log(kde.pdf(residuals) + LOG_FLOOR))
        return -log_likelihood
    except np.linalg.LinAlgError:
        return BAD_FIT_PENALTY


# =====================================================================
# 反復最尤推定（共通エンジン）
# =====================================================================

KdeObjectiveFactory = Callable[[np.ndarray], Callable[..., float]]
"""残差から目的関数を作るファクトリの型（引数: 残差）。"""


def _iterative_mle(
    x_centered: np.ndarray,
    y: np.ndarray,
    a0: float,
    b0: float,
    make_objective: KdeObjectiveFactory,
    max_iter: int = DEFAULT_MAX_ITER,
    tolerance: float = DEFAULT_TOLERANCE,
) -> tuple[float, float, float, str, int]:
    """KDE を固定 → 最適化 → 収束判定 を繰り返す共通ループ.

    ``run_mle_and_aic``（全体 KDE）と ``run_mle_and_aic_yearly_pooled``
    （年度別合成 KDE）で処理手順が完全に同一なため、目的関数の生成だけを
    差し替え可能な形で共通化している。

    Parameters
    ----------
    x_centered : array-like
        中心化済みの年度。
    y : array-like
        log 濃度。
    a0, b0 : float
        初期値（傾き・切片）。
    make_objective : callable
        残差を引数に取り、``f(params, x, y) -> float`` を返す関数を返す
        ファクトリ。
    max_iter : int, optional
        最大反復回数。
    tolerance : float, optional
        収束判定閾値（パラメータ変化量のユークリッドノルム）。

    Returns
    -------
    tuple
        ``(a1, b1, log_likelihood_1, converged_status, final_iter)``。
        ``log_likelihood_1`` は最後に実行した最適化の最大対数尤度。
    """
    a_current, b_current = a0, b0
    prev_a, prev_b = a_current, b_current

    converged_status = "未収束"
    final_iter = max_iter
    log_likelihood_1 = -np.inf

    for i in range(max_iter):
        # --- 現在のパラメータで残差を計算し、KDE を固定 ---
        residuals = y - (a_current * x_centered + b_current)
        objective = make_objective(residuals)

        # --- 固定 KDE を使った最適化 ---
        result = minimize(
            objective,
            [a_current, b_current],
            args=(x_centered, y),
            method="L-BFGS-B",
        )
        a_new, b_new = result.x
        log_likelihood_1 = -result.fun

        # --- 収束判定 ---
        param_change = np.linalg.norm([a_new - prev_a, b_new - prev_b])
        if param_change < tolerance:
            converged_status = "収束"
            final_iter = i + 1
            a1, b1 = a_new, b_new
            break

        # --- 次の iteration のために更新 ---
        prev_a, prev_b = a_current, b_current
        a_current, b_current = a_new, b_new
    else:
        # 最大回数到達
        a1, b1 = a_current, b_current
        converged_status = "最大回数到達"
        final_iter = max_iter

    return a1, b1, log_likelihood_1, converged_status, final_iter


def _make_global_kde_objective(residuals: np.ndarray) -> Callable[..., float]:
    """全体残差から単一 KDE を作り、固定 KDE の NLL を返すファクトリ."""
    kde_fixed = gaussian_kde(residuals)

    def objective(params: Sequence[float], x: np.ndarray, y: np.ndarray) -> float:
        return neg_log_likelihood_fixed_kde(params, x, y, kde_fixed)

    return objective


def _make_yearly_pooled_objective(
    x_centered: np.ndarray,
) -> KdeObjectiveFactory:
    """年度別合成 KDE の NLL を返すファクトリを生成する."""

    def factory(residuals: np.ndarray) -> Callable[..., float]:
        yearly_kdes_fixed = build_yearly_kdes(residuals, x_centered)

        def objective(params: Sequence[float], x: np.ndarray, y: np.ndarray) -> float:
            return neg_log_likelihood_yearly_fixed_kde(
                params, x, y, yearly_kdes_fixed
            )

        return objective

    return factory


def run_mle_and_aic(
    df: pd.DataFrame, max_iter: int = DEFAULT_MAX_ITER, tolerance: float = DEFAULT_TOLERANCE
) -> tuple:
    """全体 KDE を反復更新する KDE-MLE と AIC 判定.

    Returns
    -------
    tuple
        ``(a1, b1, aic0, aic1, p1, x_base, converged_status, final_iter)``。
    """
    x_raw = df["year"].values
    x_base = x_raw.min()
    x_centered = x_raw - x_base
    y = df["log_conc"].values
    n_samples=len(y)

    # 初期値（OLS）
    a_current, b_current = np.polyfit(x_centered, y, 1)

    a1, b1, ll1, converged_status, final_iter = _iterative_mle(
        x_centered,
        y,
        a_current,
        b_current,
        _make_global_kde_objective,
        max_iter=max_iter,
        tolerance=tolerance,
    )

    # --- 定数モデル（0次式） ---
    result_0 = minimize(
        neg_log_likelihood_0,
        [np.mean(y)],
        args=(x_centered, y),
        method="L-BFGS-B",
    )
    ll0 = -result_0.fun

    aic0, aic1, p1 = compute_aic_posterior(ll1, ll0)
    aicc0, aicc1, p1_c = compute_aic_posterior(ll1, ll0, n_samples)

    return a1, b1, aic0, aic1, p1, aicc0, aicc1, p1_c, x_base, converged_status, final_iter

def calc_coef_ci_bootstrap_yearly(
    x_centered: np.ndarray,
    y: np.ndarray,
    a_hat: float,
    b_hat: float,
    n_boot: int = 250,
    alpha: float = 0.05,
    seed: int = 42,
) -> tuple[tuple[float, float], tuple[float, float]]:
    """年度別合成 KDE を用いた残差ブートストラップにより傾き a と切片 b の信頼区間を算出する."""
    residuals = y - (a_hat * x_centered + b_hat)
    if np.std(residuals) < 1e-6:
        return (np.nan, np.nan), (np.nan, np.nan)

    # ★ 修正：第1引数が residuals、第2引数が x_centered
    yearly_kdes = build_yearly_kdes(residuals, x_centered)
    if not yearly_kdes:
        return (np.nan, np.nan), (np.nan, np.nan)

    # --- 高速化：1次元グリッドで対数密度を事前計算し np.interp で補間 ---
    margin = max(1.0, 3.0 * float(np.std(residuals)))
    grid = np.linspace(float(residuals.min()) - margin, float(residuals.max()) + margin, 2048)
    pdf_grid = eval_pooled_yearly_pdf(grid, yearly_kdes)
    log_pdf_grid = np.log(np.maximum(pdf_grid, 1e-10))
    edge_val = float(np.log(1e-10))

    rng = np.random.default_rng(seed)
    y_fitted = a_hat * x_centered + b_hat
    n = len(y)

    a_samples = []
    b_samples = []

    for _ in range(n_boot):
        boot_res = rng.choice(residuals, size=n, replace=True)
        y_boot = y_fitted + boot_res

        def nll_boot(params):
            a, b = params
            res_b = y_boot - (a * x_centered + b)
            log_p = np.interp(res_b, grid, log_pdf_grid, left=edge_val, right=edge_val)
            return -float(np.sum(log_p))

        res = minimize(nll_boot, [a_hat, b_hat], method="L-BFGS-B", tol=1e-4)
        if res.success:
            a_samples.append(res.x[0])
            b_samples.append(res.x[1])

    if len(a_samples) < 10:
        return (np.nan, np.nan), (np.nan, np.nan)

    low_q = 100.0 * (alpha / 2.0)
    high_q = 100.0 * (1.0 - alpha / 2.0)
    ci_a = (float(np.percentile(a_samples, low_q)), float(np.percentile(a_samples, high_q)))
    ci_b = (float(np.percentile(b_samples, low_q)), float(np.percentile(b_samples, high_q)))
    return ci_a, ci_b

def calc_coef_ci_bootstrap_yearly_old(
    x_centered: np.ndarray,
    y: np.ndarray,
    a_hat: float,
    b_hat: float,
    n_boot: int = 500,
    alpha: float = 0.05,
    seed: int = 42,
) -> tuple[tuple[float, float], tuple[float, float]]:
    """年度別合成 KDE を用いた残差ブートストラップにより傾き a と切片 b の信頼区間を算出する."""
    residuals = y - (a_hat * x_centered + b_hat)
    if np.std(residuals) < 1e-6:
        return (np.nan, np.nan), (np.nan, np.nan)

    yearly_kdes, overall_std = build_yearly_kdes(x_centered, residuals)
    if not yearly_kdes:
        return (np.nan, np.nan), (np.nan, np.nan)

    rng = np.random.default_rng(seed)
    y_fitted = a_hat * x_centered + b_hat
    n = len(y)

    a_samples = []
    b_samples = []

    for _ in range(n_boot):
        boot_res = rng.choice(residuals, size=n, replace=True)
        y_boot = y_fitted + boot_res

        def nll_boot(params):
            a, b = params
            res_b = y_boot - (a * x_centered + b)
            return -float(np.sum(eval_pooled_yearly_pdf(res_b, yearly_kdes, overall_std)))

        res = minimize(nll_boot, [a_hat, b_hat], method="L-BFGS-B")
        if res.success:
            a_samples.append(res.x[0])
            b_samples.append(res.x[1])

    if len(a_samples) < 10:
        return (np.nan, np.nan), (np.nan, np.nan)

    low_q = 100.0 * (alpha / 2.0)
    high_q = 100.0 * (1.0 - alpha / 2.0)
    ci_a = (float(np.percentile(a_samples, low_q)), float(np.percentile(a_samples, high_q)))
    ci_b = (float(np.percentile(b_samples, low_q)), float(np.percentile(b_samples, high_q)))
    return ci_a, ci_b


def run_mle_and_aic_yearly_pooled(
    df: pd.DataFrame, max_iter: int = DEFAULT_MAX_ITER, tolerance: float = DEFAULT_TOLERANCE
) -> tuple:
    """環境省（図 5）準拠：年度別残差分布の合成による反復最尤推定と AIC 判定.

    Returns
    -------
    tuple
        ``(a1, b1, aic0, aic1, p1, x_base, converged_status, final_iter)``。
    """
    x_raw = df["year"].values
    x_base = x_raw.min()
    x_centered = x_raw - x_base
    y = df["log_conc"].values
    n_samples = len(y)

    # 初期値（OLS）
    a_current, b_current = np.polyfit(x_centered, y, 1)

    a1, b1, ll1, converged_status, final_iter = _iterative_mle(
        x_centered,
        y,
        a_current,
        b_current,
        _make_yearly_pooled_objective(x_centered),
        max_iter=max_iter,
        tolerance=tolerance,
    )

    # --- 定数モデル（0次式） ---
    result_0 = minimize(
        neg_log_likelihood_yearly_0,
        [np.mean(y)],
        args=(x_centered, y),
        method="L-BFGS-B",
    )
    ll0 = -result_0.fun
    aic0, aic1, p1, aicc0, aicc1, p1_c = compute_aic_posterior(
            ll1,
            ll0,
            n_params_linear=2,
            n_params_constant=1,
            n_samples=n_samples,
        )
    

    ci_a_moe, ci_b_moe = calc_coef_ci_bootstrap_yearly(x_centered, y, a1, b1)

    return (
        a1,
        ci_a_moe,
        b1,
        ci_b_moe,
        aic0,
        aic1,
        p1,
        aicc0,
        aicc1,
        p1_c,
        x_base,
        converged_status,
        final_iter,
    )

    #aic0, aic1, p1 = compute_aic_posterior(ll1, ll0)
    #return a1, b1, aic0, aic1, p1, x_base, converged_status, final_iter


# =====================================================================
# ブートストラップ検定（前期 vs 後期）
# =====================================================================

def determine_split_year(df: pd.DataFrame) -> Optional[int]:
    """前期・後期を分割する年（年度の中央値）を返す.

    年度数が 4 未満の場合は ``None``（検定不能）。
    """
    years = df["year"].dropna().unique()
    if len(years) < 4:
        return None
    return int(np.median(years))


def compare_periods(
    df: pd.DataFrame, split_year: int = 2015, R: int = BOOTSTRAP_R
) -> np.ndarray:
    """前期・後期の平均値差をブートストラップで推定し 95% CI を返す.

    Parameters
    ----------
    df : DataFrame
        対象物質のデータ（``year`` と ``log_conc`` を含む）。
    split_year : int
        この年以下を前期、より後を後期とする。
    R : int
        リサンプリング回数。

    Returns
    -------
    ndarray
        ``[2.5, 97.5]`` パーセンタイル。前期・後期いずれかが 2 点未満なら
        ``[nan, nan]``。
    """
    A = df[df["year"] <= split_year]["log_conc"].values
    B = df[df["year"] > split_year]["log_conc"].values
    if len(A) < 2 or len(B) < 2:
        return np.array([np.nan, np.nan])

    diffs = []
    for _ in range(R):
        A_sample = resample(A, replace=True)
        B_sample = resample(B, replace=True)
        diffs.append(np.mean(A_sample) - np.mean(B_sample))

    return np.percentile(diffs, BOOTSTRAP_ALPHA_PERCENTILES)


# =====================================================================
# 二項ロジスティック回帰（ND 割合のトレンド）
# =====================================================================

def binomial_trend_analyze(df: pd.DataFrame) -> tuple[float, float, float, float]:
    """ND（検出下限未満）割合の経年傾向を二項 GLM で評価する.

    全期間の最大検出下限値を統一基準とし、それを下回る地点を ND と
    みなして年ごとの ND / 非 ND 数を集計する。

    Returns
    -------
    tuple
        ``(aic0, aic1, p1, slope)``。
    """
    unified_dl = df["detection_limit"].max()
    df["low_flag"] = (df["concentration"] < unified_dl).astype(int)

    # 年ごとに「ND 数 (low_count)」と「ND でない数 (not_low_count)」を集計
    yearly_data = df.groupby("year").agg(
        low_count=("low_flag", "sum"),
        total_count=("low_flag", "count"),
    ).reset_index()

    yearly_data["not_low_count"] = yearly_data["total_count"] - yearly_data["low_count"]

    x_year = yearly_data["year"].astype(float).to_numpy()
    X1 = sm.add_constant(x_year).astype(float)

    # 応答変数を [ND数, 非ND数] の 2 列行列として作成
    y_binomial = yearly_data[["low_count", "not_low_count"]].to_numpy().astype(float)

    # 有効なデータ（全試行数が 0 でない行）のフィルタリング
    valid_mask = (yearly_data["total_count"] > 0) & (~np.isnan(X1).any(axis=1))
    X1 = X1[valid_mask]
    y_binomial = y_binomial[valid_mask]

    X0 = np.ones((len(y_binomial), 1), dtype=float)

    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always", category=PerfectSeparationWarning)

        # freq_weights は使わず、y に 2 列の行列を渡す
        model1_fit = sm.GLM(y_binomial, X1, family=sm.families.Binomial()).fit()
        aic1 = model1_fit.aic

        model0_fit = sm.GLM(y_binomial, X0, family=sm.families.Binomial()).fit()
        aic0 = model0_fit.aic

    delta_aic = aic1 - aic0
    numerator = np.exp(-0.5 * delta_aic)
    p1 = numerator / (1 + numerator)
    slope = model1_fit.params[1] if len(model1_fit.params) > 1 else np.nan

    return aic0, aic1, p1, slope


# =====================================================================
# 前処理・1 物質ごとの解析（メイン処理の部品）
# =====================================================================

def load_and_preprocess(filepath: str, output_dir: str) -> tuple[pd.DataFrame, str]:
    """入力 CSV を読み込み、ND 補正・log 変換・欠損除去を行う.

    Parameters
    ----------
    filepath : str
        入力 CSV のパス。
    output_dir : str
        変換後データの出力先ディレクトリ。

    Returns
    -------
    tuple of (DataFrame, str)
        前処理済みデータフレームと、変換後 CSV の出力パス。
    """
    os.makedirs(output_dir, exist_ok=True)

    df_all = pd.read_csv(filepath, usecols=list(INPUT_COLUMNS))
    df_all["concentration"] = df_all["concentration"].apply(clean_value_user)
    df_all["detection_limit"] = df_all["detection_limit"].apply(clean_value_user)

    # ND（不検出）は検出下限値 / sqrt(2) で置換する
    is_nondetect = df_all["concentration"] <= df_all["detection_limit"]
    substitute_value = df_all["detection_limit"] / ND_SUBSTITUTE_DIVISOR
    df_all["concentration_sub"] = df_all["concentration"].copy()
    df_all.loc[is_nondetect, "concentration_sub"] = substitute_value[is_nondetect]

    df_all["log_conc"] = np.log10(df_all["concentration_sub"])
    df_all = df_all.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["log_conc", "detection_limit", "concentration"]
    )

    filename = os.path.splitext(os.path.basename(filepath))[0]
    output_path = os.path.join(output_dir, f"{filename}_conv.csv")
    df_all.to_csv(output_path, index=False, encoding=CSV_ENCODING)  # Excel 文字化け対策
    print(f"→ 変換後のデータを保存しました: {output_path}")

    return df_all, output_path

def analyze_substance(df: pd.DataFrame, substance: str, filename: str) -> Optional[dict]:

    # --- 補助指標の算出 ---
    n_samples = int(len(df))
    n_years = int(df["year"].nunique())
    is_nd = df["concentration"] <= df["detection_limit"]
    nd_ratio = float(is_nd.mean()) if n_samples > 0 else np.nan
    dl_min = float(df["detection_limit"].min()) if n_samples > 0 else np.nan
    dl_max = float(df["detection_limit"].max()) if n_samples > 0 else np.nan

    """1 物質について 5 つの解析を実行し、結果レコードを返す."""
    print(f"\n==================== {filename} : {substance} ====================")
    
    # ND のみのケース等は除外
    if np.std(df["log_conc"]) < MIN_LOG_CONC_STD:
        return None
    
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["year"])
    
    if len(df) < MIN_SAMPLES:
        print("データ不足のためスキップ（地点数 < 24）")
        return None

# --- ① KDE-MLE（simple） ---
    print("\n=== Running KDE-MLE (simple) ===")
    (
    a1,
    b1,
    ci_a_s,
    ci_b_s,
    aic0_1,
    aic1_1,
    p1_s,
    aicc0_1,
    aicc1_1,
    p1_c_s,
    x_base,
    ) = run_mle_and_aic_simple(df)

    trend_s = (
        f"p1 >= {P1_THRESHOLD} 傾向あり"
        if p1_s >= P1_THRESHOLD
        else f"p1 < {P1_THRESHOLD} 傾向なし"
    )
    print(f"Slope (a): {a1:.4f}, Intercept (b): {b1:.4f}, p1: {p1_s:.4f} → {trend_s}")
    print(f"ci傾き ci_a_s = { ci_a_s}, ci切片 ci_b_s = {ci_b_s}")
    print(
        f"  [AICc確認] AICc0 = {aicc0_1:.4f}, AICc1 = {aicc1_1:.4f}, "
        f"p1_c = {p1_c_s:.4f}"
    )
  
    # --- ② KDE-MLE（iterative） ---
    print("\n=== Running KDE-MLE (iterative) ===")
    try:
        (
            a,
            b,
            ci_a_i,
            ci_b_i,
            aic0,
            aic1,
            p1,
            aicc0,
            aicc1,
            p1_c,
            x_base,
            mle_status,
            mle_iters,
            tolerance,
        ) = run_mle_and_aic_iterative(df, a1, b1, MAIN_ITERATIVE_MAX_ITER)
        trend = (
            f"p1 >= {P1_THRESHOLD} 傾向あり"
            if p1 >= P1_THRESHOLD
            else f"p1 < {P1_THRESHOLD} 傾向なし"
        )
        print(f"状態: {mle_status} (反復回数: {mle_iters})")
        print(f"傾き a = {a:.4f}, 切片 b = {b:.4f}, p1 = {p1:.4f} → {trend}")
        print(
            f"傾き a = {a:.4f} ({ci_a_i[0]:.4f}, {ci_a_i[1]:.4f}), "
            f"切片 b = {b:.4f} ({ci_b_i[0]:.4f}, {ci_b_i[1]:.4f}), ")
        #print(f"  傾きci_a_i = {ci_a_i}, 切片ci_b_i = {ci_b_i}")
        print(
            f"  [AICc確認] AICc0 = {aicc0:.4f}, AICc1 = {aicc1:.4f}, "
            f"p1_c = {p1_c:.4f}"
        )
    except Exception as e:
        print(f"最尤推定エラー: {e}")
        return None
    
    # --- ③ 環境省 図 5 準拠（年度別残差合成 KDE） ---
    print("\n=== Running KDE-MLE (MOE Fig.5 Yearly Pooled) ===")
    a_moe, b_moe, p1_moe = np.nan, np.nan, np.nan
    aicc0_moe, aicc1_moe, p1_c_moe = np.nan, np.nan, np.nan
    trend_moe, status_moe, iters_moe = "エラー", "エラー", np.nan
    try:
        (
            a_moe,
            ci_a_moe,
            b_moe,
            ci_b_moe,
            _,
            _,
            p1_moe,
            aicc0_moe,
            aicc1_moe,
            p1_c_moe,
            _,
            status_moe,
            iters_moe,
        ) = run_mle_and_aic_yearly_pooled(df)
        trend_moe = (
            f"p1 >= {P1_THRESHOLD} 傾向あり"
            if p1_moe >= P1_THRESHOLD
            else f"p1 < {P1_THRESHOLD} 傾向なし"
        )
        print(f"状態: {status_moe} (反復回数: {iters_moe})")
        print(
            f"傾き a = {a_moe:.4f}, 切片 b = {b_moe:.4f}, "
            f" ci_a_moe{ ci_a_moe},  ci_b_moe{ ci_b_moe}"
            f"p1 = {p1_moe:.4f} → {trend_moe}"
        )
        print(
            f"  [AICc確認] AICc0 = {aicc0_moe:.4f}, AICc1 = {aicc1_moe:.4f}, "
            f"p1_c = {p1_c_moe:.4f}"
        )
    except Exception as e:
        print(f"年度別合成KDEエラー: {e}")


    # --- ④ ブートストラップ検定 ---
    ci_bs = np.array([np.nan, np.nan])
    bs_trend = "bs_trendエラー"
    try:
        split_year = determine_split_year(df)
        if split_year is not None:
            ci_bs = compare_periods(df, split_year=split_year)
            if ci_bs[0] > 0 and not np.isnan(ci_bs[0]):
                bs_trend = "後期は有意に低濃度"
            elif ci_bs[1] < 0 and not np.isnan(ci_bs[1]):
                bs_trend = "後期は有意に高濃度"
            else:
                bs_trend = "有意差なし"
    except Exception as e:
        print(f"ブートストラップエラー: {e}")

    # --- ⑤ 二項ロジスティック回帰 ---
    a_bin, p1_bin = np.nan, np.nan
    trend_bin = "trend_binエラー"
    try:
        df_binomial = df[["year", "concentration", "detection_limit"]].copy()
        aic0_bin, aic1_bin, p1_bin, a_bin = binomial_trend_analyze(df_binomial)
        trend_bin = "傾向あり" if p1_bin >= P1_THRESHOLD else "傾向なし"
    except Exception as e:
        print(f"二項回帰エラー: {e}")

 # すべての結果を辞書にまとめて返却
    return {
        "substance": substance,
        "year": df["year"].unique().tolist(),
        "n_samples": n_samples,
        "n_years": n_years,
        "nd_ratio": nd_ratio,
        "dl_min": dl_min,
        "dl_max": dl_max,

        # 1. KDE-MLE (simple)
        "slope_simple": a1,
        "slope_ci_simple": f"{ci_a_s[0]:.4f}, {ci_a_s[1]:.4f}", 
        "intercept_simple": b1,
        "intercept_ci_simple":  f"{ci_b_s[0]:.4f}, {ci_b_s[1]:.4f}", 
        "p1_simple": p1_s,
        "p1_c_simple": p1_c_s,
        "trend_simple": trend_s,
        # 2. KDE-MLE (iterative)
        "slope_iterative": a,
        "slope_ci_iterative": f"{ci_a_i[0]:.4f}, {ci_a_i[1]:.4f}",        
        "intercept_iterative": b,
        "intercept_ci_iterative": f"{ci_b_i[0]:.4f}, {ci_b_i[1]:.4f}", 
        "p1_iterative": p1,
        "p1_c_iterative": p1_c,
        "trend_iterative": trend,
        "mle_status": mle_status,
        "mle_iters": mle_iters,
        "tolerance": tolerance,
        # 3. KDE-MLE (MOE Fig.5 Yearly Pooled)
        "slope_moe_yearly": a_moe,
        "slope_ci_moe_yearly": f"{ci_a_moe[0]:.4f}, {ci_a_moe[1]:.4f}",   
        "intercept_moe_yearly": b_moe,
        "intercept_ci_moe_yearly": f"{ci_b_moe[0]:.4f}, {ci_b_moe[1]:.4f}",
        "p1_moe_yearly": p1_moe,
        "p1_c_moe_yearly": p1_c_moe,
        "trend_moe_yearly": trend_moe,
        "moe_yearly_status": status_moe,
        "moe_yearly_iters": iters_moe,
        # 4. Bootstrap
        "bootstrap_CI": ci_bs,
        "bs_trend": bs_trend,
        # 5. Binomial Logistic
        "binomial_slope": a_bin,
        "binomial_p1": p1_bin,
        "binomial_trend": trend_bin,
    }

def format_results(results: list[dict]) -> pd.DataFrame:
    """結果レコード群を CSV 出力用に整形して DataFrame にする."""
    for r in results:
        # 信用区間の整形
        r["bootstrap_CI"] = format_ci(r.get("bootstrap_CI"))

        # すべての数値項目を安全に丸める
        for key in (
            "nd_ratio",
            "dl_min",
            "dl_max",
            "slope_simple",
            "intercept_simple",
            "p1_simple",
            "p1_c_simple",
            "slope_iterative",
            "intercept_iterative",
            "p1_iterative",
            "p1_c_iterative",
            "slope_moe_yearly",
            "intercept_moe_yearly",
            "p1_moe_yearly",
            "p1_c_moe_yearly",
            "binomial_slope",
            "binomial_p1",
        ):
            if key in r:
                r[key] = format_float(r.get(key))

    return pd.DataFrame(results)



def run_analysis_for_file(
    filename: str, data_dir: str = "data", output_dir: str = "outputs"
) -> Optional[pd.DataFrame]:
    """1 つの入力 CSV について、全物質の解析を実行し結果を保存する.

    Returns
    -------
    DataFrame or None
        結果テーブル。入力ファイルが無い場合は ``None``。
    """
    print(f"\n\n==================== ファイル処理開始: {filename} ====================")
    filepath = os.path.join(data_dir, f"{filename}.csv")
    if not os.path.exists(filepath):
        print(f"ファイルが見つかりません: {filepath}")
        return None

    df_all, _ = load_and_preprocess(filepath, output_dir)

    substances = df_all["substance"].unique()
    print(f"対象物質: {substances}")

    results: list[dict] = []
    for substance in substances:
        df_sub = df_all[df_all["substance"] == substance].copy()
        record = analyze_substance(df_sub, substance, filename)
        if record is not None:
            results.append(record)

    df_results = format_results(results)
    print(f"\n--- 最終結果（テーブル形式）  {filename} ---")
    print(df_results.to_string(index=False))

    result_path = os.path.join(output_dir, f"{filename}_2025_moe.csv")
    df_results.to_csv(result_path, index=False, encoding=CSV_ENCODING)
    return df_results


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """コマンドライン引数を解析する."""
    parser = argparse.ArgumentParser(
        description="KDE-MLE による POPs 年次トレンド解析（環境省法）",
    )
    parser.add_argument(
        "filenames",
        nargs="*",
        default=list(DEFAULT_FILENAMES),
        help=f"解析するファイル名（拡張子なし）。既定: {' '.join(DEFAULT_FILENAMES)}",
    )
    parser.add_argument(
        "--data-dir", default="data", help="入力 CSV のディレクトリ（既定: data）"
    )
    parser.add_argument(
        "--output-dir",
        default="outputs",
        help="出力 CSV のディレクトリ（既定: outputs）",
    )
    args, unknown = parser.parse_known_args(argv)

    return args    #return parser.parse_args(argv)


def main(argv: Optional[Sequence[str]] = None) -> None:
    """エントリポイント。指定された各ファイルを順に解析する."""
    args = parse_args(argv)
    for filename in args.filenames:
        run_analysis_for_file(filename, args.data_dir, args.output_dir)


if __name__ == "__main__":
    main()



==================== ファイル処理開始: PFOS大気温暖期 ====================
→ 変換後のデータを保存しました: outputs\PFOS大気温暖期_conv.csv
対象物質: ['ペルフルオロオクタンスルホン酸（PFOS）' 'ペルフルオロオクタンスルホン酸（PFOS）2023'
 'ペルフルオロオクタンスルホン酸（PFOS）2022' 'ペルフルオロオクタンスルホン酸（PFOS）2021'
 'ペルフルオロオクタンスルホン酸（PFOS）2020' 'ペルフルオロオクタンスルホン酸（PFOS）2019'
 'ペルフルオロオクタンスルホン酸（PFOS）2017' 'ペルフルオロオクタンスルホン酸（PFOS）2016'
 'ペルフルオロオクタンスルホン酸（PFOS）2015' 'ペルフルオロオクタンスルホン酸（PFOS）2014'
 'ペルフルオロオクタンスルホン酸（PFOS）2013' 'ペルフルオロオクタンスルホン酸（PFOS）2012'
 'ペルフルオロオクタンスルホン酸（PFOS）2011']

==================== PFOS大気温暖期 : ペルフルオロオクタンスルホン酸（PFOS） ====================

=== Running KDE-MLE (simple) ===
Slope (a): -0.0074, Intercept (b): 0.5518, p1: 0.9991 → p1 >= 0.95 傾向あり
ci傾き ci_a_s = (-0.011360799868479547, -0.003392359859880658), ci切片 ci_b_s = (0.5182752939156381, 0.5853560387841998)
  [AICc確認] AICc0 = 46.3411, AICc1 = 32.3388, p1_c = 0.9991

=== Running KDE-MLE (iterative) ===
状態: 収束 (反復回数: 1)
傾き a = -0.0072, 切片 b = 0.5359, p1 = 0.9998 → p1 >= 0.95 傾向あり
傾き a = -0.0072 (-0.0101, -0.0041), 切片 b = 0